# Test: Repo Restructure (`refactor/repo-structure`)

Verify that the restructured codebase imports and runs correctly on Colab Pro.

## 0. Setup

In [ ]:
# Clone the feature branch
!rm -rf Conformal-training
!git clone --branch refactor/repo-structure https://github.com/iemppu/Conformal-training.git
%cd Conformal-training
!git log --oneline -1

In [ ]:
# Install dependencies (jax needed by smooth_conformal)
!pip install -q jax jaxlib scikit-learn tqdm

In [ ]:
import sys, os
# Ensure repo root is on path
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print(f"Repo root: {REPO_ROOT}")
print(f"Directory listing: {os.listdir('.')}")

## 1. Import Tests — Models

In [ ]:
from src.models.resnet import resnet
from src.models.vgg import vgg16, vgg19_bn
from src.models.densenet import densenet

print("resnet:", resnet)
print("vgg16:", vgg16)
print("densenet:", densenet)
print("[PASS] src.models imports OK")

## 2. Import Tests — Methods (no arg-parsing deps)

In [ ]:
from src.methods.losses import LDAMLoss, FocalLoss
print("LDAMLoss:", LDAMLoss)
print("FocalLoss:", FocalLoss)
print("[PASS] src.methods.losses OK")

In [ ]:
from src.methods import isotonic
print("isotonic:", isotonic)
print("[PASS] src.methods.isotonic OK")

In [ ]:
from src.methods import split_conformal
print("split_conformal:", split_conformal)
print("[PASS] src.methods.split_conformal OK")

In [ ]:
from src.methods.sorting_nets import comm_pattern_batcher
print("comm_pattern_batcher:", comm_pattern_batcher)
print("[PASS] src.methods.sorting_nets OK")

In [ ]:
from src.methods.variational_sorting_net import VariationalSortingNet
print("VariationalSortingNet:", VariationalSortingNet)
print("[PASS] src.methods.variational_sorting_net OK")

In [ ]:
from src.methods.smooth_conformal import smooth_aps_score, smooth_aps_score_all
print("smooth_aps_score:", smooth_aps_score)
print("[PASS] src.methods.smooth_conformal OK")

## 3. Import Tests — Methods (arg-parsing chain)

`pytorch_ops` → `numpy_ops` → `isotonic` and `pytorch_ops` also imports `config` (parser).

`scores` imports `pytorch_ops`, `smooth_conformal`, `metrics`, etc.

In [ ]:
from src.methods.numpy_ops import isotonic_l2, isotonic_kl
print("isotonic_l2:", isotonic_l2)
print("[PASS] src.methods.numpy_ops OK")

In [ ]:
# pytorch_ops triggers argparse at import time — this tests the full chain
from src.methods.pytorch_ops import soft_rank, soft_sort
print("soft_rank:", soft_rank)
print("soft_sort:", soft_sort)
print("[PASS] src.methods.pytorch_ops OK")

In [ ]:
from src.methods.scores import get_HPS_scores, compute_scores_diff, Smoothquantile
print("get_HPS_scores:", get_HPS_scores)
print("[PASS] src.methods.scores OK")

## 4. Import Tests — Data

In [ ]:
from src.data.cifar100 import load_cifar100
print("load_cifar100:", load_cifar100)
print("[PASS] src.data.cifar100 OK")

## 5. Import Tests — Utils

In [ ]:
from src.utils import config as parser_file
print("config.initialize:", parser_file.initialize)
print("[PASS] src.utils.config OK")

In [ ]:
from src.utils.metrics import evaluate_predictions, get_scores_HPS, get_scores, classwise_conformal, Marginal_conformal
print("evaluate_predictions:", evaluate_predictions)
print("[PASS] src.utils.metrics OK")

## 6. Import Tests — conformal_utils (the big one)

This file imports from models, methods, data, and utils — it's the integration test.

In [ ]:
from src.methods.conformal_utils import (
    Estimate_quantile_n, UniformMatchingLoss, PinballMarginal,
    load_train_objs, base_path_for_finetune, load_checkpoint,
    prepare_dataloader, loss_fnc, check_path, create_final_data,
    create_folder, test_model, loss_cal, create_optimizers,
    find_scores_APS, find_scores_HPS, find_scores_RAPS,
)
print("[PASS] src.methods.conformal_utils OK — all key symbols imported")

## 7. Smoke Test — Instantiate a model & run forward pass

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# ResNet-20 on CIFAR-100
model = resnet(depth=20, num_classes=100, use_fc_single=False).to(device)
x = torch.randn(4, 3, 32, 32).to(device)
out = model(x)
print(f"ResNet-20 output shape: {out.shape}")  # expect [4, 100]
assert out.shape == (4, 100)
print("[PASS] ResNet-20 forward pass OK")

In [ ]:
# DenseNet-100 on CIFAR-100
model_dn = densenet(depth=100, dropRate=0, num_classes=100, growthRate=12, compressionRate=2, use_fc_single=False).to(device)
out_dn = model_dn(x)
print(f"DenseNet-100 output shape: {out_dn.shape}")
assert out_dn.shape == (4, 100)
print("[PASS] DenseNet-100 forward pass OK")

## 8. Smoke Test — Load CIFAR-100 data

In [ ]:
train_loader, val_loader, test_loader_all, num_train, num_val, train_ds, val_ds, test_ds = \
    load_cifar100(save_path=None, n_tr=50, n_val=10, n_cal=10, n_test=10, train_rho=1.0, val_rho=1.0, num_classes=100)

print(f"Train dataset size: {len(train_ds)}")
print(f"Val dataset size:   {len(val_ds)}")
print(f"Test dataset size:  {len(test_ds)}")

# Grab one batch
imgs, labels = next(iter(train_loader))
print(f"Batch shape: {imgs.shape}, labels shape: {labels.shape}")
print("[PASS] CIFAR-100 data loading OK")

## 9. Smoke Test — Scoring functions

In [ ]:
model.eval()
with torch.no_grad():
    imgs_d = imgs.to(device)
    labels_d = labels.to(device)
    logits = model(imgs_d)
    probs = torch.nn.Softmax(dim=1)(logits)

    # HPS scores
    hps = get_HPS_scores(probs, labels_d)
    print(f"HPS scores shape: {hps.shape}, mean: {hps.mean():.4f}")

print("[PASS] Scoring functions OK")

## 10. Smoke Test — scripts/train.py argparse

In [ ]:
!python scripts/train.py --help 2>&1 | head -20
print("\n[PASS] scripts/train.py --help OK")

## Summary

In [ ]:
print("="*50)
print("All import and smoke tests passed!")
print("The restructured repo works correctly.")
print("="*50)